In [1]:
import pandas as pd

sales = pd.read_csv(
    "/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_validation.csv"
)

print(sales.shape)
sales.head()

(30490, 1919)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [2]:
row = sales.iloc[0]

print(row[:10])



id          HOBBIES_1_001_CA_1_validation
item_id                     HOBBIES_1_001
dept_id                         HOBBIES_1
cat_id                            HOBBIES
store_id                             CA_1
state_id                               CA
d_1                                     0
d_2                                     0
d_3                                     0
d_4                                     0
Name: 0, dtype: object


In [3]:
series = row.iloc[6:]

series = series.astype(float)

print(series.head())
print(series.shape)

d_1    0.0
d_2    0.0
d_3    0.0
d_4    0.0
d_5    0.0
Name: 0, dtype: float64
(1913,)


In [23]:
import os
import matplotlib.pyplot as plt

os.makedirs("visualizations", exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(series.values)

plt.title("Daily Sales")

plt.xlabel("Days")

plt.ylabel("Units Sold")

plt.show()

In [5]:
non_zero_days = (series > 0).sum()

total_days = len(series)

print("Non-zero sales days:", non_zero_days)

print("Total days:", total_days)

print("Percentage active:", 
      round((non_zero_days / total_days) * 100, 2), "%")

Non-zero sales days: 421
Total days: 1913
Percentage active: 22.01 %


In [6]:
sales_only = sales.iloc[:, 6:]

active_counts = (sales_only > 0).sum(axis=1)

sales["active_percentage"] = (
    active_counts / sales_only.shape[1]
) * 100


In [7]:
top_products = sales.sort_values(
    by="active_percentage",
    ascending=False
)

top_products[
    [
        "id",
        "active_percentage"
    ]
].head(10)

,id,active_percentage
5859,FOODS_3_586_CA_2_validation,99.843178
17721,FOODS_3_252_TX_2_validation,99.790904
8877,FOODS_3_555_CA_3_validation,99.790904
18024,FOODS_3_555_TX_2_validation,99.790904
8711,FOODS_3_389_CA_3_validation,99.738630
5353,FOODS_3_080_CA_2_validation,99.738630
8908,FOODS_3_586_CA_3_validation,99.738630
14500,FOODS_3_080_TX_1_validation,99.738630
8402,FOODS_3_080_CA_3_validation,99.738630
8550,FOODS_3_228_CA_3_validation,99.686357


In [8]:
target_row = sales[
    sales["id"] == "FOODS_3_586_CA_2_validation"
]

target_row = target_row.iloc[0]

In [9]:
series = target_row.iloc[6:-1]

series = series.astype(float)

print(series.head())

print(series.shape)

d_1    34.0
d_2    25.0
d_3    16.0
d_4    25.0
d_5    37.0
Name: 5859, dtype: float64
(1913,)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(series.values)

plt.title("FOODS_3_586_CA_2 Sales")

plt.xlabel("Days")

plt.ylabel("Units Sold")

plt.show()

In [11]:
train = series[:-28]

test = series[-28:]

print("Train size:", len(train))

print("Test size:", len(test))

Train size: 1885
Test size: 28


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(train.values, label="Train")

plt.plot(
    range(len(train), len(series)),
    test.values,
    label="Test"
)

plt.legend()

plt.title("Train/Test Split")

plt.show()

In [13]:
series.index = pd.RangeIndex(
    start=0,
    stop=len(series)
)

train = series[:-28]

test = series[-28:]

In [14]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    train,
    order=(1,1,1),
    seasonal_order=(1,1,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

result = model.fit()

In [15]:
print(result.summary())

                                     SARIMAX Results                                     
Dep. Variable:                              5859   No. Observations:                 1885
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 7)   Log Likelihood               -6639.739
Date:                           Sun, 31 May 2026   AIC                          13289.478
Time:                                   16:19:12   BIC                          13317.142
Sample:                                        0   HQIC                         13299.671
                                          - 1885                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1938      0.023      8.437      0.000       0.149       0.239
ma.L1         -0.9009      0.010    -87.722

In [16]:
forecast = result.forecast(steps=28)

print(forecast.head())

1885    35.423138
1886    32.570740
1887    32.651008
1888    34.121800
1889    38.951295
Name: predicted_mean, dtype: float64


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.plot(
    test.index,
    test.values,
    label="Actual"
)

plt.plot(
    test.index,
    forecast.values,
    label="Forecast"
)

plt.legend()

plt.title("SARIMA Forecast vs Actual")

plt.show()

In [18]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

mae = mean_absolute_error(
    test,
    forecast
)

rmse = mean_squared_error(
    test,
    forecast
) ** 0.5

print("MAE:", mae)

print("RMSE:", rmse)

MAE: 7.207803393029889
RMSE: 8.368226882447239


In [ ]:
plt.figure(figsize=(15,5))

plt.plot(series.values)

plt.title("Complete Sales History")

plt.xlabel("Days")

plt.ylabel("Units Sold")

plt.show()

In [ ]:
rolling_mean = series.rolling(window=30).mean()

plt.figure(figsize=(15,5))

plt.plot(series.values, alpha=0.4, label="Daily Sales")

plt.plot(
    rolling_mean.values,
    label="30-Day Rolling Mean"
)

plt.legend()

plt.title("Sales Trend with Rolling Average")

plt.show()

In [ ]:
weekly_pattern = []

for i in range(7):
    weekly_pattern.append(
        series[i::7].mean()
    )

days = [
    "Day 1",
    "Day 2",
    "Day 3",
    "Day 4",
    "Day 5",
    "Day 6",
    "Day 7"
]

plt.figure(figsize=(10,5))

plt.bar(days, weekly_pattern)

plt.title("Average Weekly Sales Pattern")

plt.ylabel("Average Units Sold")

plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    train.index[-100:],
    train.values[-100:],
    label="Train"
)

plt.plot(
    test.index,
    test.values,
    label="Actual"
)

plt.plot(
    forecast.index,
    forecast.values,
    label="Forecast"
)

plt.legend()

plt.title("SARIMA Forecasting")

plt.show()